# Diabetes trajectory clustering

이 노트북은 `data/data.csv`를 넣은 뒤 설정 파일만 바꿔서 clustering을 실행하는 예제입니다.

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

def find_repo_root():
    candidates = [Path.cwd(), Path.cwd() / "diabetes-trajectory-clustering", *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "diabetes_trajectory_clustering").exists() and (candidate / "configs").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find the diabetes-trajectory-clustering repository folder.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

for module_name in list(sys.modules):
    if module_name == "diabetes_trajectory_clustering" or module_name.startswith("diabetes_trajectory_clustering."):
        del sys.modules[module_name]

from diabetes_trajectory_clustering import run_from_config

CLUSTER_VARS = ["HBA1C", "BMI", "HOMA_IR", "HOMA_B"]

def make_output_dir(mode, cluster_vars):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    var_label = "_".join(str(var).replace("/", "-").replace("\\", "-") for var in cluster_vars)
    out_dir = REPO_ROOT / "outputs" / mode / f"{stamp}_{var_label}"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"data.csv exists = {(REPO_ROOT / 'data' / 'data.csv').exists()}")
print(f"CLUSTER_VARS = {CLUSTER_VARS}")


## 1. Trajectory 기반 clustering

In [ ]:
trajectory_out_dir = make_output_dir("trajectory", CLUSTER_VARS)

result = run_from_config(
    REPO_ROOT / "configs" / "example_trajectory_config.json",
    data_path=REPO_ROOT / "data" / "data.csv",
    out_dir=trajectory_out_dir,
    cluster_base_vars=CLUSTER_VARS,
)

print(f"Saved to: {trajectory_out_dir}")

result["cluster_sizes"]

In [ ]:
result["k_evaluation"]

## 2. 진단 시점 기반 clustering

진단 당시 또는 진단 직전 값으로 clustering하려면 아래 셀을 실행합니다.

In [ ]:
diagnosis_out_dir = make_output_dir("diagnosis", CLUSTER_VARS)

diagnosis_result = run_from_config(
    REPO_ROOT / "configs" / "example_diagnosis_config.json",
    data_path=REPO_ROOT / "data" / "data.csv",
    out_dir=diagnosis_out_dir,
    cluster_base_vars=CLUSTER_VARS,
)

print(f"Saved to: {diagnosis_out_dir}")

diagnosis_result["cluster_sizes"]